In [1]:
# %%
import os
import sys
import json
import re # <-- Added
from pathlib import Path

# --- Make sure your utils are imported for pathing ---
file_path = os.getcwd()
project_root = os.path.dirname(os.path.dirname(file_path))
print(f"Project Root: {project_root}")
if project_root not in sys.path:
    sys.path.append(project_root)

from dotenv import load_dotenv
import chromadb
from langchain_core.embeddings import Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma 
from langchain_core.documents import Document

# --- Import your project's agents and utils ---
from crunch_llm.code.agents import BasicAgent 
from crunch_llm.code.utils import initialize_local_data


Project Root: /oscar/data/gk/jdtoscan/LLM/Crunch_LLM


/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
analyzer_system_prompt_scic = """
You are an expert Scientific Computing and Numerical Methods code analyst. Your task is to analyze the provided code snippet (often Julia or Python) and extract key information into a structured JSON format. You must be highly specific and infer the values from the code structures, library calls (like Trixi.jl, DiffEq), and variable names.

Your final output must be a single, valid JSON object. Do not add any explanatory text outside the JSON.

### Fields to Extract

1.  **`method`**: Identify the specific spatial discretization or numerical scheme.
    * *Examples*: `Discontinuous Galerkin (DGSEM)`, `Finite Volume (FV)`, `WENO-5`, `Spectral Collocation`, `Compact Finite Difference`.
    * *Hints*: Look for keywords like `DGSEM`, `LobattoLegendreBasis` (implies Spectral Element), `FluxReconstruction`, or finite difference stencils.

2.  **`equation`**: Identify the specific governing equations.
    * *Examples*: `Compressible Navier-Stokes`, `Euler Equations`, `Maxwell's Equations`, `Advection-Diffusion`.
    * *Hints*: Look for physics structs like `CompressibleEulerEquations2D`, `HyperbolicDiffusion`, or variable names like `rho, rho_u, E`.

3.  **`dimensionality`**: Specify the spatial and temporal dimensions.
    * *Examples*: `1D`, `2D`, `3D`.
    * *Hints*: Look for mesh constructors (`trees_per_dimension`), coordinate tuples `(x, y)`, or array shapes.

4.  **`solver_or_optimizer`**: Identify the time-integration scheme or the linear solver/optimizer.
    * *Examples*: `Low-Storage Runge-Kutta (LSRK)`, `Crank-Nicolson`, `RK45`, `Newton-Krylov`, `SSBroyden`.
    * *Hints*: Look for `OrdinaryDiffEq` calls (e.g., `RDPK3SpFSAL49`, `Tsit5`), `solve(...)` arguments, or iterative solver definitions.

5.  **`enhancements`**: List specific advanced numerical features found in the code.
    * *Examples*: `["Adaptive Mesh Refinement (AMR)"]`, `["Shock Capturing (Hybrid DG-FV)"]`, `["Entropy Stable Flux"]`, `["Löhner Error Indicator"]`.
    * *Hints*: Look for `AMRCallback`, `VolumeIntegralShockCapturing`, `IndicatorHennemannGassner`, or specific stabilization terms. If none, use `[]`.

6.  **`BCs`**: Identify the boundary conditions.
    * *Examples*: `Dirichlet (No-Slip)`, `Neumann (Adiabatic)`, `Periodic`, `Riemann Invariant`.
    * *Hints*: Look for function definitions like `boundary_condition_slip_wall`, `NoSlip`, or periodicity flags `periodicity=(false, true)`.

7.  **`problem`**: Describe the high-level physical scenario in one sentence.
    * *Examples*: `Simulate the Lid-Driven Cavity flow`, `Solve the Kelvin-Helmholtz instability`, `Wave propagation in a heterogeneous medium`.

8.  **`description`**: Provide a concise 1-2 sentence summary of the code's methodology.

---

### Example 1: High-Order DG with AMR (Trixi.jl Style)

*Input Code Snippet:*
```julia
equations = CompressibleEulerEquations2D(1.4)
indicator_sc = IndicatorHennemannGassner(equations, basis, alpha_max=0.5)
volume_integral = VolumeIntegralShockCapturingHG(indicator_sc; volume_flux_dg=flux_ranocha, volume_flux_fv=flux_lax_friedrichs)
dg = DGSEM(polydeg=3, surface_flux=flux_hll, volume_integral=volume_integral)

amr_controller = ControllerThreeLevel(semi, IndicatorLöhner(semi), max_level=6)
amr_callback = AMRCallback(semi, amr_controller)

sol = solve(ode, RDPK3SpFSAL49(); dt=1.0)
```

*Your Required JSON Output:*
```json
{{
  "method": "Discontinuous Galerkin (DGSEM)",
  "equation": "Compressible Euler Equations",
  "dimensionality": "2D",
  "solver_or_optimizer": "Low-Storage Runge-Kutta (LSRK - RDPK3SpFSAL49)",
  "enhancements": [
    "Adaptive Mesh Refinement (AMR)",
    "Hybrid Shock Capturing (Hennemann-Gassner)",
    "Entropy Stable Flux (Ranocha)"
  ],
  "BCs": "Unknown",
  "problem": "Simulate compressible flow with shock capturing and adaptive mesh refinement.",
  "description": "A high-order DGSEM simulation of the 2D Compressible Euler equations. It features hybrid FV shock capturing and dynamic AMR using a Löhner error indicator."
}}
```

### Example 2: Classical Finite Difference (Python Style)

*Input Code Snippet:*
```python
# Discretization using 4th order Central Difference
def laplacian(u, dx):
    return ( -u[2:] + 16*u[1:-1] - 30*u + 16*u[0:-2] - u[:-4] ) / (12 * dx**2)

# Time stepping
def step(u):
    # Forward Euler for advection
    u_new = u - dt * c * (u[1:] - u[:-1])/dx + dt * nu * laplacian(u, dx)
    return u_new
```

*Your Required JSON Output:*
```json
{{
  "method": "Finite Difference (4th Order Central)",
  "equation": "Advection-Diffusion",
  "dimensionality": "1D",
  "solver_or_optimizer": "Forward Euler",
  "enhancements": [],
  "BCs": "N/A",
  "problem": "Solve the linear advection-diffusion equation.",
  "description": "A simple 1D finite difference solver using 4th order central differences for diffusion and Forward Euler for time integration."
}}
```
"""

In [3]:

# 1. Define the exact path where you want to save the database
my_database_path ="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_NM"

COLLECTION_NAME = 'Numerical_Methods_opt'

# 2. Pass that path to your function
vector_store = initialize_local_data(db_path=my_database_path,collection_name=COLLECTION_NAME)

analyzer_agent = BasicAgent(
                     model='gemini-2.5-flash', 
                     model_provider='google',
                     system_prompt=analyzer_system_prompt_scic,
                     verbose=False)


Initializing with Ollama. Model: bge-m3:567m
✅ Vector store initialized successfully from path: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_NM
   Provider: ollama, Model: bge-m3:567m, Collection: Numerical_Methods_opt


In [6]:
import os
import re
import json
from pathlib import Path
from langchain_core.documents import Document

# The directory containing your code files
target_path = '/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/Numerical_Methods/Scripts/'

all_results = {}
print(f"\nStarting analysis of all files in: {target_path}")

# os.walk will traverse the entire directory tree
for root, dirs, files in os.walk(target_path):
    for filename in files:
        file_path = os.path.join(root, filename)
        
        # Skip non-python/julia files (Added .py just in case)
        if not filename.endswith('.jl') and not filename.endswith('.py'):
            continue
            
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                code_content = f.read()
            line_count = code_content.count('\n') + 1
            print(f"Processing: {filename} ({line_count} lines)...")
            
            # --- 1. Run Analyzer Agent ONCE on the full file ---
            print("  - Running Analyzer Agent...")
            
            # Call Agent (Assuming it returns a dict with 'output')
            agent_result = analyzer_agent.run(code_content)
            
            # Handle response types (Dict vs String)
            if isinstance(agent_result, dict) and 'output' in agent_result:
                summary_string = agent_result['output'].strip()
            else:
                summary_string = str(agent_result).strip()
            
            # --- START FIX: Parse JSON and Flatten List ---
            try:
                # Clean markdown fences if the agent added them
                clean_json = summary_string.replace("```json", "").replace("```", "").strip()
                data = json.loads(clean_json)

                analysis_result = {
                    # Cast everything to string and flatten lists
                    "method": str(data.get("method", "Unknown")),
                    "equation": str(data.get("equation", "Unknown")),
                    "dimensionality": str(data.get("dimensionality", "Unknown")),
                    "solver_or_optimizer": str(data.get("solver_or_optimizer", "Unknown")),
                    
                    # CRITICAL FIX: Join the list into a single string
                    "enhancements": ", ".join(data.get("enhancements", [])),
                    
                    "BCs": str(data.get("BCs", "Unknown")),
                    "problem": str(data.get("problem", "Unknown")),
                    "description": str(data.get("description", "Unknown")),
                    "filename": filename
                }
            except Exception as e:
                print(f"  - Warning: Could not parse JSON '{summary_string[:50]}...'. Error: {e}")
                analysis_result = {
                    "method": "ParseError",
                    "equation": "Unknown",
                    "enhancements": "",
                    "filename": filename
                }
            # --- END FIX ---

            print(f"  - Analysis: {analysis_result['method']} | {analysis_result['equation']}")
            
            
            # --- 2. Correct Splitting & Re-assembly ---
            print("  - Splitting code into cells...")
            
            # This pattern matches '# ## Cell N:' at the START of any line
            split_pattern = r'(^\s*[#\s]+Cell\s*\d+:.*)'
            
            # Split using MULTILINE mode
            parts = re.split(split_pattern, code_content, flags=re.MULTILINE)
            
            cleaned_cells = []
            
            # Handle the "prologue" (anything before the first Cell marker)
            if parts[0].strip():
                cleaned_cells.append(parts[0].strip())
                
            # Loop over the split parts, taking 2 at a time
            for i in range(1, len(parts), 2):
                if i + 1 < len(parts):
                    full_cell = parts[i] + parts[i+1]
                    cleaned_cells.append(full_cell.strip())
                else:
                    cleaned_cells.append(parts[i].strip())

            num_cells = len(cleaned_cells)
            
            print(f"  - Found {num_cells} cells. Ingesting each as a 'part'...")

            # --- 3. Prepare common metadata & header info ---
            base_metadata = {
                'source_file': file_path,
                'filename': filename,
                'type': 'code-notebook',
                **analysis_result # Add the flattened analysis
            }
            
            file_ext = Path(file_path).suffix.lower()
            language_map = {'.py': 'python', '.m': 'matlab', '.jl': 'julia', '.cpp': 'cpp', '.c': 'c'}
            language = language_map.get(file_ext, 'unknown')
            comment_styles = {'python': '#', 'matlab': '%', 'julia': '#', 'c': '//', 'cpp': '//'}
            comment = comment_styles.get(language, '#')
            
            # Create a pretty string for the header (for reading)
            analysis_str = json.dumps(analysis_result, indent=2).replace('{', '').replace('}', '').replace('"', '')

            docs_to_add = []
            
            # --- 4. Loop through each cell and create a Document ---
            for i, cell_content in enumerate(cleaned_cells):
                part_num = i + 1
                
                # --- Extract the cell's description ---
                cell_description = "File Prologue / Imports" 
                
                if cell_content.startswith('# ## Cell'):
                     first_line = cell_content.split('\n', 1)[0]
                     cell_description = first_line.strip().lstrip('#').strip()
                else:
                    first_line = ""
                    for line in cell_content.split('\n'):
                        if line.strip() and not line.strip().startswith('#'):
                            first_line = line
                            break
                    if first_line:
                        cell_description = first_line.strip()
                
                # --- Create metadata for this chunk ---
                chunk_metadata = base_metadata.copy()
                chunk_metadata['part'] = f"{part_num}/{num_cells}"
                chunk_metadata['cell_description'] = cell_description
                
                # --- Create the full header for this chunk ---
                header = f"""{comment} CODE ANALYSIS (PART {part_num} of {num_cells})
{comment} DESCRIPTION:
{comment} 
{comment} {analysis_str.replace(chr(10), f'{chr(10)}{comment} ')}
{comment} =======================================================\n\n"""
                
                enhanced_content = header + cell_content
                
                doc = Document(page_content=enhanced_content, metadata=chunk_metadata)
                docs_to_add.append(doc)

            # --- 5. Add all documents for this file at once ---
            if docs_to_add:
                vector_store.add_documents(docs_to_add)
                
            print(f"✅ Ingested: {filename} ({num_cells} chunks)")
            all_results[file_path] = {"analysis": analysis_result, "chunks_created": num_cells}

        except Exception as e:
            print(f"--- UNHANDLED ERROR analyzing {filename}: {e} ---")
            all_results[file_path] = {"error": str(e)}

# --- 6. Save and Display Results ---
output_file = 'full_code_analysis.json'
print(f"\nAnalysis complete! ✅ Saving all results to {output_file}...")
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=4)
print("Done.")


Starting analysis of all files in: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/Numerical_Methods/Scripts/
Processing: Burgers_MoC.jl (237 lines)...
  - Running Analyzer Agent...
  - Analysis: Method of Characteristics (MoC) | Burgers' Equation (Scalar Conservation Law)
  - Splitting code into cells...
  - Found 6 cells. Ingesting each as a 'part'...
✅ Ingested: Burgers_MoC.jl (6 chunks)
Processing: CG_2D.jl (819 lines)...
  - Running Analyzer Agent...
  - Analysis: Mixed Finite Element Method (Taylor-Hood Pk/Pk-1) | Steady Incompressible Navier-Stokes
  - Splitting code into cells...
  - Found 13 cells. Ingesting each as a 'part'...
✅ Ingested: CG_2D.jl (13 chunks)
Processing: DG_1D.jl (247 lines)...
  - Running Analyzer Agent...
  - Analysis: Discontinuous Galerkin (DGSEM) | Inviscid Burgers' Equation
  - Splitting code into cells...
  - Found 7 cells. Ingesting each as a 'part'...
✅ Ingested: DG_1D.jl (7 chunks)
Processing: Fourier_Spectral.jl (254 lines)...
  - Run

In [7]:

# Assuming 'vector_store' is your initialized Chroma object
print("Fetching all documents from the vector store...")

# The .get() method returns a dictionary with IDs, documents, metadatas, etc.
# We specify what information we want to retrieve.
all_entries = vector_store.get(
    include=["metadatas", "documents"] # You can also include "embeddings"
)

# Let's print the first 5 documents found
num_entries = len(all_entries['ids'])
print(f"Found {num_entries} total entries.\n")

for i in range(min(5, num_entries)):
    doc_id = all_entries['ids'][i]
    metadata = all_entries['metadatas'][i]
    content = all_entries['documents'][i]
    
    print(f"--- Entry {i+1} ---")
    print(f"ID: {doc_id}")
    print(f"Source File: {metadata.get('source_file', 'N/A')}")
    print(f"Content Preview: {content[:15000]}...") # Print first 150 chars
    print("-" * 20)


Fetching all documents from the vector store...
Found 45 total entries.

--- Entry 1 ---
ID: a8b36a59-58d1-4bba-b354-da7c8fa9e91b
Source File: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/Numerical_Methods/Scripts/Burgers_MoC.jl
Content Preview: # CODE ANALYSIS (PART 1 of 6)
# DESCRIPTION:
# 
# 
#   method: Method of Characteristics (MoC),
#   equation: Burgers' Equation (Scalar Conservation Law),
#   dimensionality: 1D,
#   solver_or_optimizer: Secant Method (Root Finding),
#   enhancements: Shock Fitting (Rankine-Hugoniot), Equal Area Rule,
#   BCs: Periodic,
#   problem: Generate the exact entropy solution for the 1D Burgers' equation with a sinusoidal initial condition.,
#   description: This code implements the Method of Characteristics to compute the exact entropy solution for the 1D Burgers' equation. It uses implicit characteristic tracing combined with shock fitting via the Rankine-Hugoniot condition (Equal Area Rule) to handle shock formation, providing a high

In [5]:
# import chromadb
# import os
# from pathlib import Path
# import sys

# # --- Make sure your utils are imported for pathing ---
# # (This isn't strictly necessary but good for consistency)
# file_path = os.getcwd()
# project_root = os.path.dirname(file_path)
# if project_root not in sys.path:
#     sys.path.append(project_root)

# my_collection_name = COLLECTION_NAME

# print(f"Attempting to clear elements from collection: '{my_collection_name}'")
# print(f"From database path: {my_database_path}")

# try:
#     # --- 2. INITIALIZE THE CLIENT ---
#     client = chromadb.PersistentClient(path=my_database_path)
    
#     # --- 3. GET THE COLLECTION ---
#     collection = client.get_collection(name=my_collection_name)
    
#     # --- 4. GET ALL ELEMENT IDS ---
#     # We get all items. include=[] is efficient as we only need IDs.
#     all_items = collection.get(include=[]) 
#     all_ids = all_items['ids']
    
#     if not all_ids:
#         print(f"\nInfo: Collection '{my_collection_name}' is already empty.")
#     else:
#         # --- 5. DELETE ALL ELEMENTS BY ID ---
#         print(f"\nFound {len(all_ids)} elements. Deleting all elements...")
#         collection.delete(ids=all_ids)
#         print(f"✅ Success: All {len(all_ids)} elements have been deleted from '{my_collection_name}'.")
#         print("The collection itself still exists.")

# except chromadb.errors.CollectionNotFoundError:
#     print(f"\nInfo: Collection '{my_collection_name}' does not exist. Nothing to delete.")
# except Exception as e:
#     print(f"\nAn error occurred: {e}")
#     print("Please check the path and ChromaDB server status.")



Attempting to clear elements from collection: 'Numerical_Methods_opt'
From database path: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_NM

Found 44 elements. Deleting all elements...
✅ Success: All 44 elements have been deleted from 'Numerical_Methods_opt'.
The collection itself still exists.
